In [5]:
import json, yaml
import pandas as pd
from pprint import pprint


resp_path = "/Users/nident/Desktop/JOB/startup/agent_scorer/data/responses/simple_model_response_20260427_180511.json"
with open(resp_path, 'r') as f:
    resp = json.load(f)

gt_path = resp['metadata']['criterion_path']

with open(gt_path, 'r') as f:
    gt_data = yaml.safe_load(f)

final_resp = json.loads(resp['final']['raw_response'])


my_map = {"entailment": 1, "neutral": 0, "contradiction": -1}

In [6]:
def collect_results(gt_data, final_resp, verbose=False):
    labels = [-1, 0, 1]
    label_names = {
        -1: "contradiction",
        0: "neutral",
        1: "entailment",
    }

    gold = gt_data["gold"]
    points = final_resp["points"]
    gt_count = len(gold)
    pred_count = len(points)
    paired_count = min(gt_count, pred_count)

    if verbose and gt_count != pred_count:
        print(f"Length mismatch: gold={gt_count}, points={pred_count}, paired={paired_count}")

    y_true = []
    y_pred = []

    for idx, (gt, point) in enumerate(zip(gold, points)):
        gt_label = my_map[gt["label"]]
        model_label = int(point["score"])

        y_true.append(gt_label)
        y_pred.append(model_label)

        if verbose:
            print(idx)
            print(f"GT: {gt_label}, Model: {model_label}")

    total = len(y_true)

    # rows = ground truth, columns = model prediction
    confusion_matrix = {
        true_label: {pred_label: 0 for pred_label in labels}
        for true_label in labels
    }

    for true_label, pred_label in zip(y_true, y_pred):
        confusion_matrix[true_label][pred_label] += 1

    per_class = {}

    for label in labels:
        tp = confusion_matrix[label][label]
        fp = sum(confusion_matrix[other][label] for other in labels if other != label)
        fn = sum(confusion_matrix[label][other] for other in labels if other != label)
        tn = total - tp - fp - fn

        precision = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_class[label_names[label]] = {
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "support": sum(confusion_matrix[label].values()),
        }

    accuracy = sum(
        1 for true_label, pred_label in zip(y_true, y_pred)
        if true_label == pred_label
    ) / total if total > 0 else 0.0

    macro_precision = sum(v["precision"] for v in per_class.values()) / len(labels)
    macro_recall = sum(v["recall"] for v in per_class.values()) / len(labels)
    macro_f1 = sum(v["f1_score"] for v in per_class.values()) / len(labels)

    weighted_f1 = (
        sum(v["f1_score"] * v["support"] for v in per_class.values()) / total
        if total > 0
        else 0.0
    )

    return {
        "total": total,
        "gt_count": gt_count,
        "pred_count": pred_count,
        "paired_count": paired_count,
        "length_mismatch": gt_count != pred_count,
        "missing_predictions": max(gt_count - pred_count, 0),
        "extra_predictions": max(pred_count - gt_count, 0),
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1_score": macro_f1,
        "weighted_f1_score": weighted_f1,
        "confusion_matrix": confusion_matrix,
        "per_class": per_class,
    }


In [7]:
res = collect_results(gt_data, final_resp)
pprint(res)

{'accuracy': 1.0,
 'confusion_matrix': {-1: {-1: 1, 0: 0, 1: 0},
                      0: {-1: 0, 0: 1, 1: 0},
                      1: {-1: 0, 0: 0, 1: 1}},
 'extra_predictions': 0,
 'gt_count': 3,
 'length_mismatch': False,
 'macro_f1_score': 1.0,
 'macro_precision': 1.0,
 'macro_recall': 1.0,
 'missing_predictions': 0,
 'paired_count': 3,
 'per_class': {'contradiction': {'FN': 0,
                                 'FP': 0,
                                 'TN': 2,
                                 'TP': 1,
                                 'f1_score': 1.0,
                                 'precision': 1.0,
                                 'recall': 1.0,
                                 'support': 1},
               'entailment': {'FN': 0,
                              'FP': 0,
                              'TN': 2,
                              'TP': 1,
                              'f1_score': 1.0,
                              'precision': 1.0,
                              'recall': 

In [ ]:
from pathlib import Path


def parse_final_response(resp):
    final = resp["final"]
    if isinstance(final, dict) and "raw_response" in final:
        return json.loads(final["raw_response"])
    if isinstance(final, str):
        return json.loads(final)
    return final


def metrics_from_confusion_matrix(confusion_matrix):
    labels = [-1, 0, 1]
    label_names = {-1: "contradiction", 0: "neutral", 1: "entailment"}
    total = sum(confusion_matrix[t][p] for t in labels for p in labels)

    per_class = {}
    for label in labels:
        tp = confusion_matrix[label][label]
        fp = sum(confusion_matrix[other][label] for other in labels if other != label)
        fn = sum(confusion_matrix[label][other] for other in labels if other != label)
        tn = total - tp - fp - fn

        precision = tp / (tp + fp) if tp + fp > 0 else 0.0
        recall = tp / (tp + fn) if tp + fn > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

        per_class[label_names[label]] = {
            "TP": tp,
            "TN": tn,
            "FP": fp,
            "FN": fn,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "support": sum(confusion_matrix[label].values()),
        }

    accuracy = sum(confusion_matrix[label][label] for label in labels) / total if total > 0 else 0.0
    macro_precision = sum(v["precision"] for v in per_class.values()) / len(labels)
    macro_recall = sum(v["recall"] for v in per_class.values()) / len(labels)
    macro_f1 = sum(v["f1_score"] for v in per_class.values()) / len(labels)
    weighted_f1 = sum(v["f1_score"] * v["support"] for v in per_class.values()) / total if total > 0 else 0.0

    return {
        "total": total,
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1_score": macro_f1,
        "weighted_f1_score": weighted_f1,
        "confusion_matrix": confusion_matrix,
        "per_class": per_class,
    }


def aggregate_results(results):
    labels = [-1, 0, 1]
    confusion_matrix = {true: {pred: 0 for pred in labels} for true in labels}

    for item in results:
        matrix = item["metrics"]["confusion_matrix"]
        for true in labels:
            for pred in labels:
                confusion_matrix[true][pred] += matrix[true][pred]

    return metrics_from_confusion_matrix(confusion_matrix)


def mismatch_summary(results):
    mismatches = [item for item in results if item["metrics"]["length_mismatch"]]
    return {
        "files_with_length_mismatch": len(mismatches),
        "missing_predictions": sum(item["metrics"]["missing_predictions"] for item in mismatches),
        "extra_predictions": sum(item["metrics"]["extra_predictions"] for item in mismatches),
        "paired_points_in_mismatch_files": sum(item["metrics"]["paired_count"] for item in mismatches),
    }


def collect_batch_results(responses_dir="/Users/nident/Desktop/JOB/startup/agent_scorer/data/responses/simple_model"):
    responses_dir = Path(responses_dir)
    results = []
    errors = []

    for resp_path in sorted(responses_dir.glob("*.json")):
        try:
            with resp_path.open("r", encoding="utf-8") as f:
                resp = json.load(f)

            gt_path = Path(resp["metadata"]["criterion_path"])
            with gt_path.open("r", encoding="utf-8") as f:
                gt_data = yaml.safe_load(f)

            final_resp = parse_final_response(resp)
            metrics = collect_results(gt_data, final_resp)
            results.append({
                "response_file": resp_path.name,
                "criterion_file": gt_path.name,
                "metrics": metrics,
            })
        except Exception as exc:
            errors.append({"response_file": resp_path.name, "error": str(exc)})

    overall = aggregate_results(results) if results else None
    mismatch_stats = mismatch_summary(results)

    rows = [
        {
            "response_file": item["response_file"],
            "criterion_file": item["criterion_file"],
            "total": item["metrics"]["total"],
            "gt_count": item["metrics"]["gt_count"],
            "pred_count": item["metrics"]["pred_count"],
            "length_mismatch": item["metrics"]["length_mismatch"],
            "missing_predictions": item["metrics"]["missing_predictions"],
            "extra_predictions": item["metrics"]["extra_predictions"],
            "accuracy": item["metrics"]["accuracy"],
            "macro_f1_score": item["metrics"]["macro_f1_score"],
            "weighted_f1_score": item["metrics"]["weighted_f1_score"],
        }
        for item in results
    ]

    batch_df = pd.DataFrame(rows)
    mismatch_df = batch_df[batch_df["length_mismatch"]].copy() if not batch_df.empty else batch_df

    return results, overall, mismatch_stats, batch_df, mismatch_df, pd.DataFrame(errors)


In [9]:
batch_results, overall_stats, mismatch_stats, batch_df, mismatch_df, batch_errors = collect_batch_results()

print("files:", len(batch_results))
print("errors:", len(batch_errors))
print("mismatch files:", mismatch_stats["files_with_length_mismatch"])
pprint(mismatch_stats)
pprint(overall_stats)

mismatch_df.head()


files: 114
errors: 0
mismatch files: 0
{'extra_predictions': 0,
 'files_with_length_mismatch': 0,
 'missing_predictions': 0,
 'paired_points_in_mismatch_files': 0}
{'accuracy': 0.879245283018868,
 'confusion_matrix': {-1: {-1: 226, 0: 21, 1: 10},
                      0: {-1: 20, 0: 396, 1: 32},
                      1: {-1: 5, 0: 40, 1: 310}},
 'macro_f1_score': 0.8806155794717991,
 'macro_precision': 0.882533670767497,
 'macro_recall': 0.8788484799849682,
 'per_class': {'contradiction': {'FN': 31,
                                 'FP': 25,
                                 'TN': 778,
                                 'TP': 226,
                                 'f1_score': 0.8897637795275589,
                                 'precision': 0.900398406374502,
                                 'recall': 0.8793774319066148,
                                 'support': 257},
               'entailment': {'FN': 45,
                              'FP': 42,
                              'TN': 663,


,response_file,criterion_file,total,gt_count,pred_count,length_mismatch,missing_predictions,extra_predictions,accuracy,macro_f1_score,weighted_f1_score
